# arange-fancy-index-cross-entropy composite — cx23: rearrange then fancy-index — pick per-sample logit on (C, B) form

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `arange-fancy-index-cross-entropy`, `einops-rearrange`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "arange-fancy-index-cross-entropy"
DD_ATOM_IDS = ["arange-fancy-index-cross-entropy", "einops-rearrange"]
DD_SUBTOPICS = ["Loss: arange fancy-index cross-entropy", "Einops: Rearrange"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA code sometimes stores logits as `(C, B)` (class-first) and sometimes as `(B, C)` (batch-first). The arange-fancy-index idiom is written for batch-first: `logits[arange(B), target]`. If your incoming tensor is class-first, you `rearrange(logits, 'c b -> b c')` first and THEN apply the idiom.

Composing these two atoms exercises a real ARENA bug-magnet: indexing the wrong axis. If you forget the rearrange and write `logits[arange(B), target]` on a (C, B) tensor, you'll index the CLASS axis with arange(B) — silently wrong (or shape-error if B != C).

### Composite Exercise — rearrange then fancy-index — pick per-sample logit on (C, B) form

**Atoms exercised together**: `arange-fancy-index-cross-entropy`, `einops-rearrange`

Implement `cx23_pick_after_rearrange(logits_cb, target)` that:

- Takes `logits_cb` of shape `(C, B)` (class-first) and `target` of shape `(B,)`.
- Uses `einops.rearrange` to swap to batch-first `(B, C)`.
- Picks per-sample target logits via `logits_bc[arange(B), target]`, returning shape `(B,)`.

The output must match what you'd get by applying the same pick to the batch-first form directly. No transpose tricks (`.T`) — use `rearrange` explicitly, that's the atom being exercised.

In [ ]:
def cx23_pick_after_rearrange(logits_cb, target):
    # einops-rearrange: swap class-first to batch-first.
    logits_bc = rearrange(logits_cb, 'c b -> b c')
    # arange-fancy-index on the canonical (B, C) shape.
    B = logits_bc.shape[0]
    return logits_bc[t.arange(B), target]


<details><summary>Show solution — cx23</summary>

```python
def cx23_pick_after_rearrange(logits_cb, target):
    # einops-rearrange: swap class-first to batch-first.
    logits_bc = rearrange(logits_cb, 'c b -> b c')
    # arange-fancy-index on the canonical (B, C) shape.
    B = logits_bc.shape[0]
    return logits_bc[t.arange(B), target]
```

The two-step pattern — `rearrange` to a canonical shape, then apply the canonical idiom — is everywhere in ARENA. `einops.rearrange` is preferred over `.T` / `.permute` because the axis names document the intent: the next reader can SEE that the function expects batch-first downstream. If `C == B` (e.g. self-attention with C=seq_len, B=batch=C), forgetting the rearrange would still type-check but be silently wrong — that's why the test uses non-square shapes for the larger case.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx23'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx23',
        'subtopics': ["Loss: arange fancy-index cross-entropy", "Einops: Rearrange"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()